# UniProtKB Parquet Datalake — Demo Notebook

This notebook demonstrates querying the UniProtKB Parquet datalake using multiple engines:

1. **DuckDB** — SQL-native analytical queries directly on Parquet
2. **Polars** — Lazy-evaluation DataFrame engine with predicate pushdown
3. **Pandas** — The ubiquitous DataFrame library
4. **PyArrow** — Low-level columnar access and metadata inspection
5. **HTTP Server** — Query the lake remotely without downloading files
6. **Datalake vs REST** — Queries that are hard/impossible with the UniProt REST API

## Lake Architecture

The lake follows a **star schema** with Hive-style partitioning:

```
results/uniprot_lake/
├── core/       # Metadata, organism, gene names, annotations
├── seqs/       # Amino acid sequences
└── features/   # Unnested positional annotations (domains, sites, etc.)
    └── review_status={reviewed,unreviewed}/
        └── tax_division={human,bacteria,viruses,...}/
            └── data_0.parquet
```

Partitioning by `review_status` and `tax_division` enables **partition pruning** —
queries that filter on these columns skip irrelevant files entirely.

## Setup

In [ ]:
from pathlib import Path

# Point to your lake — adjust if running from a different location
LAKE = Path('results/uniprot_lake_test')
CORE    = str(LAKE / 'core'    / '**' / '*.parquet')
SEQS    = str(LAKE / 'seqs'    / '**' / '*.parquet')
FEATURES = str(LAKE / 'features' / '**' / '*.parquet')

print(f'Lake root: {LAKE.resolve()}')
print(f'Parquet files: {len(list(LAKE.rglob("*.parquet")))}')

---
## 1. DuckDB — SQL on Parquet

DuckDB reads Parquet natively with zero-copy, Hive partition pruning, and predicate pushdown.
No data loading step needed — just point SQL at the files.

In [ ]:
import duckdb

con = duckdb.connect()

# Register the three tables as views for convenience
con.execute(f"CREATE OR REPLACE VIEW core     AS SELECT * FROM read_parquet('{CORE}',     hive_partitioning=true)")
con.execute(f"CREATE OR REPLACE VIEW seqs     AS SELECT * FROM read_parquet('{SEQS}',     hive_partitioning=true)")
con.execute(f"CREATE OR REPLACE VIEW features AS SELECT * FROM read_parquet('{FEATURES}', hive_partitioning=true)")

print('Views registered: core, seqs, features')

### 1a. Schema overview

In [ ]:
con.sql('DESCRIBE core').df()

In [ ]:
con.sql('DESCRIBE seqs').df()

In [ ]:
con.sql('DESCRIBE features').df()

### 1b. Row counts by partition

In [ ]:
con.sql("""
    SELECT review_status, tax_division, count(*) AS n
    FROM core
    GROUP BY ALL
    ORDER BY review_status, n DESC
""").df()

### 1c. Partition pruning in action

When filtering on partition columns, DuckDB skips files that don't match — no I/O wasted.

In [ ]:
# Only touches core/review_status=reviewed/tax_division=human/
con.sql("""
    SELECT acc, id, organism_name, gene_name, seq_len, seq_mass
    FROM core
    WHERE review_status = 'reviewed'
      AND tax_division = 'human'
    ORDER BY seq_len DESC
    LIMIT 10
""").df()

### 1d. Star schema join — core + seqs + features

In [ ]:
# Join all three tables for reviewed human proteins
con.sql("""
    SELECT
        c.acc,
        c.gene_name,
        c.seq_len,
        length(s.sequence) AS seq_chars,
        count(f.type) AS n_features,
        list_distinct(list(f.type)) AS feature_types
    FROM core c
    JOIN seqs s ON c.acc = s.acc
        AND c.review_status = s.review_status
        AND c.tax_division = s.tax_division
    LEFT JOIN features f ON c.acc = f.acc
        AND c.review_status = f.review_status
        AND c.tax_division = f.tax_division
    WHERE c.review_status = 'reviewed'
      AND c.tax_division = 'human'
    GROUP BY c.acc, c.gene_name, c.seq_len, s.sequence
    ORDER BY n_features DESC
""").df()

---
## 2. Polars — Lazy DataFrames with Predicate Pushdown

Polars' `scan_parquet` creates a lazy query plan that pushes filters and column selections
down to the Parquet reader. Only the data you need is ever loaded into memory.

In [ ]:
import polars as pl

# Lazy scan — nothing is read yet
core_lf = pl.scan_parquet(CORE, hive_partitioning=True)
seqs_lf = pl.scan_parquet(SEQS, hive_partitioning=True)
features_lf = pl.scan_parquet(FEATURES, hive_partitioning=True)

print('Schema (core):')
print(core_lf.collect_schema())

### 2a. Filtered query with projection pushdown

In [ ]:
# Only the columns & rows we need are read from disk
viral_proteins = (
    core_lf
    .filter(pl.col('tax_division') == 'viruses')
    .select('acc', 'id', 'organism_name', 'gene_name', 'seq_len')
    .sort('seq_len', descending=True)
    .head(10)
    .collect()
)
viral_proteins

### 2b. Lazy join and aggregation

In [ ]:
# Feature type distribution for reviewed proteins
feature_counts = (
    features_lf
    .filter(pl.col('review_status') == 'reviewed')
    .group_by('type')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .collect()
)
feature_counts

### 2c. Inspect the query plan

In [ ]:
# Polars shows you exactly what it will do — note FILTER and PROJECT pushdown
plan = (
    core_lf
    .filter(
        (pl.col('review_status') == 'reviewed') &
        (pl.col('tax_division') == 'human')
    )
    .select('acc', 'gene_name', 'seq_len')
)
print(plan.explain())

---
## 3. Pandas — The Familiar Interface

Pandas reads Parquet via PyArrow under the hood. Good for interactive exploration,
but loads everything into memory — best used on filtered subsets.

In [ ]:
import pandas as pd

# Read a specific partition directly (avoids scanning the whole lake)
human_core = pd.read_parquet(
    LAKE / 'core' / 'review_status=reviewed' / 'tax_division=human'
)
human_core.info()
human_core.head()

### 3a. Quick summary stats

In [ ]:
# Sequence length statistics for reviewed human proteins
human_core[['seq_len', 'seq_mass']].describe()

### 3b. Pandas + glob for cross-partition reads

In [ ]:
import glob

# Read all reviewed partitions
reviewed_files = glob.glob(str(LAKE / 'core' / 'review_status=reviewed' / '*' / '*.parquet'))
reviewed_df = pd.concat([pd.read_parquet(f) for f in reviewed_files], ignore_index=True)

print(f'Reviewed proteins: {len(reviewed_df)}')
reviewed_df.groupby('organism_name')['acc'].count().sort_values(ascending=False).head(10)

---
## 4. PyArrow — Low-Level Columnar Access

PyArrow gives you direct access to Parquet metadata, row groups, column statistics,
and the underlying Arrow columnar format. Ideal for inspecting the physical structure
of the lake.

In [ ]:
import pyarrow.parquet as pq

# Pick one file to inspect
sample_file = LAKE / 'core' / 'review_status=reviewed' / 'tax_division=human' / 'data_0.parquet'
pf = pq.ParquetFile(sample_file)

print(f'Parquet version: {pf.metadata.format_version}')
print(f'Created by: {pf.metadata.created_by}')
print(f'Num row groups: {pf.metadata.num_row_groups}')
print(f'Num rows: {pf.metadata.num_rows}')
print(f'Num columns: {pf.metadata.num_columns}')
print(f'\nSchema:\n{pf.schema_arrow}')

### 4a. Row group statistics

Parquet stores min/max statistics per column per row group. Query engines use these
to skip row groups that can't contain matching data.

In [ ]:
# Inspect row group 0 column statistics
rg = pf.metadata.row_group(0)
print(f'Row group 0: {rg.num_rows} rows\n')

for i in range(min(rg.num_columns, 10)):  # first 10 columns
    col = rg.column(i)
    stats = col.statistics
    print(f'{col.path_in_schema:30s}  compression={col.compression}  '
          f'encoded_size={col.total_compressed_size:,}B')
    if stats and stats.has_min_max:
        print(f'{"":30s}  min={stats.min}  max={stats.max}')

### 4b. Dataset API with partition discovery

In [ ]:
import pyarrow.dataset as ds

# PyArrow dataset automatically discovers Hive partitions
core_ds = ds.dataset(
    str(LAKE / 'core'),
    format='parquet',
    partitioning='hive'
)

print(f'Total files: {len(core_ds.files)}')
print(f'Partitioning: {core_ds.partitioning}')
print(f'\nFiles:')
for f in sorted(core_ds.files):
    print(f'  {f}')

In [ ]:
# Filter at the dataset level — only reads matching partitions
scanner = core_ds.scanner(
    filter=(ds.field('review_status') == 'reviewed') & (ds.field('tax_division') == 'human'),
    columns=['acc', 'gene_name', 'seq_len', 'seq_mass']
)
table = scanner.to_table()
print(f'{table.num_rows} rows, {table.num_columns} cols')
table.to_pandas()

### 4c. Compression analysis

In [ ]:
# Analyze compression ratios across the lake
total_compressed = 0
total_uncompressed = 0

for parquet_file in LAKE.rglob('*.parquet'):
    meta = pq.read_metadata(parquet_file)
    for rg_idx in range(meta.num_row_groups):
        rg = meta.row_group(rg_idx)
        for col_idx in range(rg.num_columns):
            col = rg.column(col_idx)
            total_compressed += col.total_compressed_size
            total_uncompressed += col.total_uncompressed_size

ratio = total_uncompressed / total_compressed if total_compressed else 0
print(f'Total compressed:   {total_compressed / 1024 / 1024:.2f} MB')
print(f'Total uncompressed: {total_uncompressed / 1024 / 1024:.2f} MB')
print(f'Compression ratio:  {ratio:.1f}x (zstd)')

---
## 5. HTTP Server — Query the Lake Remotely

DuckDB can read Parquet files over HTTP using `httpfs`. This means you can host
the lake on any static file server (S3, GCS, nginx, Python's http.server) and
query it remotely — **only downloading the byte ranges it needs**.

### 5a. Start a local HTTP server

In [ ]:
import os
import time
import socket
import threading
from http.server import HTTPServer, SimpleHTTPRequestHandler


class RangeRequestHandler(SimpleHTTPRequestHandler):
    """HTTP handler that supports Range requests (required by Polars & DuckDB httpfs)."""

    def do_GET(self):
        range_header = self.headers.get('Range')
        if not range_header:
            return super().do_GET()

        # Translate URL path to filesystem path
        path = self.translate_path(self.path)
        if not os.path.isfile(path):
            self.send_error(404)
            return

        file_size = os.path.getsize(path)
        # Parse 'bytes=start-end'
        byte_range = range_header.replace('bytes=', '')
        parts = byte_range.split('-')
        start = int(parts[0]) if parts[0] else 0
        end = int(parts[1]) if parts[1] else file_size - 1
        end = min(end, file_size - 1)
        length = end - start + 1

        self.send_response(206)
        self.send_header('Content-Type', 'application/octet-stream')
        self.send_header('Content-Length', str(length))
        self.send_header('Content-Range', f'bytes {start}-{end}/{file_size}')
        self.send_header('Accept-Ranges', 'bytes')
        self.end_headers()

        with open(path, 'rb') as f:
            f.seek(start)
            self.wfile.write(f.read(length))

    def do_HEAD(self):
        path = self.translate_path(self.path)
        if os.path.isfile(path):
            file_size = os.path.getsize(path)
            self.send_response(200)
            self.send_header('Content-Length', str(file_size))
            self.send_header('Accept-Ranges', 'bytes')
            self.end_headers()
        else:
            self.send_error(404)

    def log_message(self, format, *args):
        pass  # suppress logging


# Find a free port
def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

PORT = find_free_port()

# Serve from the lake directory
_prev_cwd = os.getcwd()
os.chdir(LAKE.resolve())
httpd = HTTPServer(('localhost', PORT), RangeRequestHandler)
thread = threading.Thread(target=httpd.serve_forever, daemon=True)
thread.start()
os.chdir(_prev_cwd)

BASE_URL = f'http://localhost:{PORT}/results/uniprot_lake_test/'
print(f'HTTP server with Range support running at {BASE_URL}')
print(f'Lake available at {BASE_URL}/core/, {BASE_URL}/seqs/, {BASE_URL}/features/')

### 5b. Query over HTTP with DuckDB httpfs

In [ ]:
# Create a separate connection for HTTP queries
http_con = duckdb.connect()
http_con.execute("INSTALL httpfs")
http_con.execute("LOAD httpfs")

# Query the lake over HTTP — DuckDB only downloads the byte ranges it needs!
# This is exactly how it would work against S3 or any cloud object store.
http_con.sql(f"""
    SELECT acc, gene_name, organism_name, seq_len
    FROM read_parquet('{BASE_URL}/core/review_status=reviewed/tax_division=human/data_0.parquet')
    ORDER BY seq_len DESC
""").df()

### 5c. Remote aggregation — no full download

Even for aggregations across partitions, DuckDB uses HTTP range requests to read
only the column chunks it needs. The full file is never downloaded.

In [ ]:
# Aggregate across all reviewed partitions over HTTP
# Only the seq_len column bytes are fetched
http_con.sql(f"""
    SELECT
        avg(seq_len) AS avg_len,
        min(seq_len) AS min_len,
        max(seq_len) AS max_len,
        count(*)     AS n
    FROM read_parquet('{BASE_URL}/core/review_status=reviewed/tax_division=human/data_0.parquet')
""").df()

### 5d. Polars can also read over HTTP

In [ ]:
# Polars scan_parquet also supports HTTP URLs
remote_df = pl.read_parquet(
    f'{BASE_URL}/core/review_status=reviewed/tax_division=human/data_0.parquet'
)
remote_df.select('acc', 'gene_name', 'seq_len').head()

In [ ]:
# Clean up the HTTP server
httpd.shutdown()
print('HTTP server stopped')

---
## 6. Datalake vs REST — Queries That Show the Power

The UniProt REST API is excellent for fetching individual entries or simple searches,
but many analytical queries are **slow, paginated, rate-limited, or simply impossible**
via REST. Here the datalake shines.

### 6a. Cross-proteome aggregations

*"What is the average sequence length per taxonomic division?"*

Via REST you'd need to paginate through every entry and compute client-side.
Via the lake, it's a single query with partition pruning.

In [ ]:
con.sql("""
    SELECT
        tax_division,
        review_status,
        count(*)            AS n_proteins,
        round(avg(seq_len)) AS avg_seq_len,
        min(seq_len)        AS min_seq_len,
        max(seq_len)        AS max_seq_len,
        round(avg(seq_mass)) AS avg_mass_da
    FROM core
    GROUP BY tax_division, review_status
    ORDER BY n_proteins DESC
""").df()

### 6b. Feature density analysis across organisms

*"Which proteins have the highest annotation density (features per residue)?"*

REST has no aggregation endpoint. You'd need to fetch each entry individually
and count features client-side.

In [ ]:
con.sql("""
    SELECT
        c.acc,
        c.gene_name,
        c.organism_name,
        c.seq_len,
        count(f.type) AS n_features,
        round(count(f.type) / c.seq_len, 3) AS features_per_residue
    FROM core c
    JOIN features f ON c.acc = f.acc
        AND c.review_status = f.review_status
        AND c.tax_division = f.tax_division
    GROUP BY c.acc, c.gene_name, c.organism_name, c.seq_len
    HAVING count(f.type) >= 5
    ORDER BY features_per_residue DESC
    LIMIT 15
""").df()

### 6c. Sequence content analysis at scale

*"Find proteins with unusually high cysteine content (potential disulfide-rich proteins)."*

REST can't filter on sequence composition. You'd need to download every sequence
and scan them locally.

In [ ]:
con.sql("""
    SELECT
        s.acc,
        c.gene_name,
        c.organism_name,
        c.seq_len,
        -- Count cysteines in the sequence
        length(s.sequence) - length(replace(s.sequence, 'C', '')) AS n_cys,
        round(
            (length(s.sequence) - length(replace(s.sequence, 'C', '')))
            * 100.0 / length(s.sequence), 2
        ) AS pct_cys
    FROM seqs s
    JOIN core c ON s.acc = c.acc
        AND s.review_status = c.review_status
        AND s.tax_division = c.tax_division
    WHERE length(s.sequence) > 50  -- skip tiny fragments
    ORDER BY pct_cys DESC
    LIMIT 15
""").df()

### 6d. Feature co-occurrence analysis

*"Which feature types tend to co-occur on the same protein?"*

This is a self-join on the features table — completely impossible via REST.

In [ ]:
con.sql("""
    WITH feature_pairs AS (
        SELECT
            a.type AS type_a,
            b.type AS type_b,
            a.acc
        FROM features a
        JOIN features b
            ON a.acc = b.acc
            AND a.review_status = b.review_status
            AND a.tax_division = b.tax_division
            AND a.type < b.type  -- avoid duplicates
    )
    SELECT
        type_a,
        type_b,
        count(DISTINCT acc) AS n_proteins,
        count(*) AS n_co_occurrences
    FROM feature_pairs
    GROUP BY type_a, type_b
    HAVING count(DISTINCT acc) >= 2
    ORDER BY n_proteins DESC, n_co_occurrences DESC
    LIMIT 15
""").df()

### 6e. Overlapping feature detection

*"Find features that physically overlap on the same protein."*

Positional interval queries are a classic example of something that requires
a proper query engine — REST can't do interval overlap detection.

In [ ]:
con.sql("""
    SELECT
        a.acc,
        a.type AS feature_a,
        a.start_pos AS a_start, a.end_pos AS a_end,
        b.type AS feature_b,
        b.start_pos AS b_start, b.end_pos AS b_end,
        -- Overlap length
        least(a.end_pos, b.end_pos) - greatest(a.start_pos, b.start_pos) + 1 AS overlap_len
    FROM features a
    JOIN features b
        ON a.acc = b.acc
        AND a.review_status = b.review_status
        AND a.tax_division = b.tax_division
        AND a.type < b.type
        AND a.start_pos <= b.end_pos
        AND b.start_pos <= a.end_pos
    WHERE a.start_pos IS NOT NULL
      AND b.start_pos IS NOT NULL
    ORDER BY overlap_len DESC
    LIMIT 15
""").df()

### 6f. Sequence length distribution (histogram)

*"What does the sequence length distribution look like across the whole proteome?"*

No REST endpoint gives you distributions. Here we get a full histogram in milliseconds.

In [ ]:
hist_df = con.sql("""
    SELECT
        floor(seq_len / 100) * 100 AS len_bucket,
        count(*) AS n_proteins
    FROM core
    GROUP BY len_bucket
    ORDER BY len_bucket
""").df()

# Quick ASCII histogram
max_count = hist_df['n_proteins'].max()
for _, row in hist_df.iterrows():
    bar_len = int(row['n_proteins'] / max_count * 40)
    print(f"{int(row['len_bucket']):>6}-{int(row['len_bucket'])+99:<6} | {'█' * bar_len} {int(row['n_proteins'])}")

### 6g. Multi-protein comparison with window functions

*"Rank proteins by sequence length within each organism, and show the top 3 per species."*

Window functions are a SQL superpower with no REST equivalent.

In [ ]:
con.sql("""
    WITH ranked AS (
        SELECT
            acc,
            gene_name,
            organism_name,
            seq_len,
            tax_division,
            row_number() OVER (
                PARTITION BY organism_name
                ORDER BY seq_len DESC
            ) AS rank_in_organism
        FROM core
        WHERE organism_name IS NOT NULL
    )
    SELECT * FROM ranked
    WHERE rank_in_organism <= 3
    ORDER BY organism_name, rank_in_organism
""").df()

### 6h. Amino acid composition profiling

*"Compare average amino acid composition between reviewed and unreviewed human proteins."*

This is a regex-based sequence analysis query — completely out of scope for REST.

In [ ]:
# Use Polars for this — its expression engine handles string ops elegantly
aa_analysis = (
    seqs_lf
    .filter(pl.col('tax_division') == 'human')
    .with_columns([
        pl.col('sequence').str.count_matches(aa).alias(f'n_{aa}')
        for aa in ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L',
                    'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
    ])
    .with_columns(
        pl.col('sequence').str.len_chars().alias('total')
    )
    .group_by('review_status')
    .agg([
        pl.col(f'n_{aa}').sum() for aa in 'ACDEFGHIKLMNPQRSTVWY'
    ] + [pl.col('total').sum()])
    .collect()
)

# Convert to percentages
for aa in 'ACDEFGHIKLMNPQRSTVWY':
    aa_analysis = aa_analysis.with_columns(
        (pl.col(f'n_{aa}') * 100.0 / pl.col('total')).round(2).alias(f'pct_{aa}')
    )

pct_cols = [f'pct_{aa}' for aa in 'ACDEFGHIKLMNPQRSTVWY']
aa_analysis.select(['review_status'] + pct_cols)

### 6i. Feature coverage heatmap data

*"What percentage of each protein's sequence is covered by annotated features?"*

Requires joining features with sequences and computing coverage — multi-step
analytical logic that REST can't express.

In [ ]:
con.sql("""
    WITH coverage AS (
        SELECT
            f.acc,
            c.gene_name,
            c.organism_name,
            c.seq_len,
            -- Approximate coverage: sum of feature spans
            -- (may overcount overlapping features)
            sum(CASE
                WHEN f.end_pos IS NOT NULL AND f.start_pos IS NOT NULL
                THEN f.end_pos - f.start_pos + 1
                ELSE 0
            END) AS total_feature_span,
            count(*) AS n_features
        FROM features f
        JOIN core c ON f.acc = c.acc
            AND f.review_status = c.review_status
            AND f.tax_division = c.tax_division
        GROUP BY f.acc, c.gene_name, c.organism_name, c.seq_len
    )
    SELECT
        acc,
        gene_name,
        organism_name,
        seq_len,
        n_features,
        total_feature_span,
        round(total_feature_span * 100.0 / seq_len, 1) AS pct_coverage
    FROM coverage
    WHERE seq_len > 0
    ORDER BY pct_coverage DESC
    LIMIT 15
""").df()

---
## 7. Bonus: Export & Interop

### 7a. DuckDB → Polars (zero-copy via Arrow)

In [ ]:
# DuckDB query result → Polars DataFrame with zero-copy Arrow transfer
arrow_table = con.sql("""
    SELECT acc, gene_name, seq_len, tax_division
    FROM core
    WHERE review_status = 'reviewed'
""").arrow()

polars_df = pl.from_arrow(arrow_table)
print(f'Zero-copy transfer: {polars_df.shape[0]} rows')
polars_df.head()

### 7b. Export query results to CSV, JSON, or new Parquet

In [ ]:
# Export a filtered subset for downstream analysis
con.sql("""
    COPY (
        SELECT acc, gene_name, organism_name, seq_len, seq_mass
        FROM core
        WHERE review_status = 'reviewed'
    ) TO '/tmp/reviewed_proteins.csv' (HEADER, DELIMITER ',')
""")

# Verify
import os
size = os.path.getsize('/tmp/reviewed_proteins.csv')
print(f'Exported to /tmp/reviewed_proteins.csv ({size:,} bytes)')
!head -5 /tmp/reviewed_proteins.csv

### 7c. DuckDB as a persistent database

You can also create a persistent DuckDB file with materialized views
for repeated access without re-scanning Parquet.

In [ ]:
# Create a persistent DuckDB database from the lake
db_path = '/tmp/uniprot_demo.duckdb'
pcon = duckdb.connect(db_path)

pcon.sql(f"""
    CREATE OR REPLACE TABLE core AS
    SELECT * FROM read_parquet('{CORE}', hive_partitioning=true)
""")

pcon.sql(f"""
    CREATE OR REPLACE TABLE features AS
    SELECT * FROM read_parquet('{FEATURES}', hive_partitioning=true)
""")

# Now queries hit the DuckDB native storage — even faster for repeated use
result = pcon.sql('SELECT count(*) AS n FROM core').fetchone()
print(f'Materialized {result[0]} rows into {db_path}')
print(f'Database size: {os.path.getsize(db_path) / 1024:.1f} KB')

pcon.close()

---

## Summary

| Engine | Best For | Partition Pruning | Lazy Eval | HTTP/S3 |
|--------|----------|:-----------------:|:---------:|:-------:|
| **DuckDB** | Complex SQL, joins, window functions, aggregations | Yes | Yes | Yes (httpfs) |
| **Polars** | DataFrame pipelines, string ops, ML preprocessing | Yes | Yes | Yes |
| **Pandas** | Interactive exploration, plotting, small subsets | Manual | No | Via PyArrow |
| **PyArrow** | Metadata inspection, schema work, zero-copy interop | Yes | Dataset API | Yes |

The datalake approach gives you **millisecond analytical queries** over data that would take
minutes or hours to retrieve via REST — with the full power of SQL, DataFrames, and columnar
storage at your fingertips.